# Model-Loader Smoke Test + Batched Throughput Probe

Verifies that each model snapshot produced by `prep_model.ipynb` loads correctly
with **FlashAttention 2 + bf16 + KV cache** on the target hardware (NVIDIA RTX PRO
6000 Blackwell, 96 GB VRAM, **internet OFF**), then measures empirical batch
ceilings so future modules can set evidence-based defaults.

**Prereqs (Kaggle notebook settings):**
- Accelerator: **GPU** (RTX PRO 6000 Blackwell)
- Internet: **Off**
- Inputs: attach **all three** committed `prep_model` versions (one per model)
  via `+ Add Input → Notebooks`. To smoke-test only a subset, edit
  `MODELS_TO_TEST` in cell 2 and attach only those prep versions.

**What it does for each target in `MODELS_TO_TEST`:**
1. Walk `/kaggle/input/` to locate the snapshot directory.
2. Load tokenizer + model with FA2 + bf16 + KV cache.
3. Print load time, parameter count, VRAM.
4. Single-batch smoke: generate 32 tokens (after a 4-token warmup), print tok/s.
5. **Batched throughput probe**: try `batches=[1, 8, 32, 64, 128]`, same prompt
   repeated, generate 32 tokens, log tok/s + peak VRAM. Stops at the first OOM.
6. Free the model before loading the next one.

**Important caveat**: the batch ceiling reported here is for *autoregressive
generation* (KV cache grows per step). It will differ from the ceilings for
forward-only extraction or per-layer activation caching — those need their own
probes when their modules land.

## 1. Configuration

In [ ]:
import os

# Offline: don't let HF clients try to call out
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

# Subset to test. Comment out any model whose prep version isn't attached.
MODELS_TO_TEST = ["llama", "aya", "qwen"]

INPUT_ROOT = "/kaggle/input"
PROMPT = "The capital of France is"
MAX_NEW_TOKENS = 32

## 2. Bootstrap — locate the repo on `/kaggle/input/`

The prep notebook clones the repo into `/kaggle/working/layerselection-repo/`, which becomes
part of the committed prep snapshot. Walk `/kaggle/input/` to find it and add the
repo root to `sys.path` so we can import `models.load`.

In [ ]:
import sys
from pathlib import Path

_marker = next(Path(INPUT_ROOT).rglob("models/load.py"), None)
assert _marker is not None, (
    f"could not find models/load.py under {INPUT_ROOT}; "
    f"is a prep_model notebook attached as input?"
)
REPO_ROOT = _marker.parent.parent  # .../layerselection-repo
sys.path.insert(0, str(REPO_ROOT))
print(f"repo root: {REPO_ROOT}")

## 3. Environment check

In [ ]:
import torch, transformers

print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"cuda:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f"device:       {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]})")
    print(f"vram total:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    import flash_attn
    print(f"flash-attn:   {flash_attn.__version__}")
except Exception as e:
    print(f"flash-attn:   NOT IMPORTABLE ({e})")
    print("              install with `pip install flash-attn>=2.8.4` if smoke-test fails on FA2")

## 4. Load + smoke-generate each model

In [ ]:
import gc, time, traceback
from models.load import (
    find_model_dir, load_model, report_load_stats,
    smoke_generate, batched_throughput_probe,
)

# Empirical observation on Blackwell 102 GB: batch=128 used only ~0.4 GB more
# than batch=1 with 32-token gen, so ceiling is far higher. Probe up to 2048.
BATCH_SIZES = (1, 8, 32, 128, 256, 512, 1024, 2048)

results = {}

for target in MODELS_TO_TEST:
    print(f"\n============================================================")
    print(f"  {target}")
    print(f"============================================================")
    model = None
    tokenizer = None
    try:
        model_dir = find_model_dir(target, INPUT_ROOT)
        print(f"  snapshot dir:     {model_dir}")

        torch.cuda.reset_peak_memory_stats()
        t_load_start = time.time()
        tokenizer, model = load_model(model_dir)
        load_stats = report_load_stats(model, t_load_start)

        print(f"\n  -- single-batch smoke --")
        gen_stats = smoke_generate(model, tokenizer,
                                   prompt=PROMPT, max_new_tokens=MAX_NEW_TOKENS)

        print(f"\n  -- batched throughput probe --")
        probe = batched_throughput_probe(model, tokenizer,
                                         prompt=PROMPT,
                                         batch_sizes=BATCH_SIZES,
                                         max_new_tokens=MAX_NEW_TOKENS)

        results[target] = {
            "snapshot_dir": str(model_dir),
            **load_stats,
            **gen_stats,
            "probe": probe,
        }
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        results[target] = {"error": f"{type(e).__name__}: {e}"}
    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n\n============================================================")
print("  Summary")
print("============================================================")
for target, r in results.items():
    if "error" in r:
        print(f"  {target:6s}  FAILED: {r['error']}")
        continue
    successful = [p for p in r.get("probe", []) if p.get("ok")]
    if successful:
        best = max(successful, key=lambda p: p["tok_per_s"])
        max_batch = max(p["batch"] for p in successful)
        oom_batch = next((p["batch"] for p in r["probe"] if not p.get("ok")), None)
        oom_note = f" (OOM at batch={oom_batch})" if oom_batch else " (no OOM in probe range)"
        print(f"  {target:6s}  load {r['elapsed_s']:5.1f}s  "
              f"single-batch {r['tok_per_s']:5.1f} tok/s  |  "
              f"max ok batch={max_batch}{oom_note}  "
              f"best throughput batch={best['batch']} → {best['tok_per_s']:.1f} tok/s "
              f"(peak {best['peak_gb']:.1f} GB)")
    else:
        print(f"  {target:6s}  load {r['elapsed_s']:5.1f}s  no batch succeeded in probe")